In [4]:
!pip install -q -U transformers datasets trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.2 MB/s eta 0:00:00


In [5]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B"
OUTPUT_DIR = "./patchpilot-lora-v1"

def generate_synthetic_training_data() -> Dataset:
    raw_data = [
        {
            "error_log": "Bandit: B608: Hardcoded SQL string detected.",
            "buggy_code": "def get_user(db, user_id):\n    query = f'SELECT * FROM users WHERE id = {user_id}'\n    return db.execute(query)",
            "patch": "--- a/db.py\n+++ b/db.py\n@@ -1,3 +1,3 @@\n def get_user(db, user_id):\n-    query = f'SELECT * FROM users WHERE id = {user_id}'\n-    return db.execute(query)\n+    query = 'SELECT * FROM users WHERE id = %s'\n+    return db.execute(query, (user_id,))"
        },
        {
            "error_log": "TypeError: Cannot read properties of undefined (reading 'map')",
            "buggy_code": "function renderList(items) {\n    return items.map(i => `<li>${i}</li>`);\n}",
            "patch": "--- a/ui.js\n+++ b/ui.js\n@@ -1,3 +1,3 @@\n function renderList(items) {\n-    return items.map(i => `<li>${i}</li>`);\n+    if (!items || !Array.isArray(items)) return '';\n+    return items.map(i => `<li>${i}</li>`);\n }"
        },
        {
            "error_log": "Out of bounds memory access (Segmentation fault)",
            "buggy_code": "int arr[5];\nfor(int i=0; i<=5; i++) {\n    arr[i] = 0;\n}",
            "patch": "--- a/main.c\n+++ b/main.c\n@@ -1,3 +1,3 @@\n int arr[5];\n-for(int i=0; i<=5; i++) {\n+for(int i=0; i<5; i++) {\n     arr[i] = 0;\n }"
        }
    ]

    formatted = []
    for item in raw_data:
        text = f"""<|im_start|>system
You are PatchPilot, an autonomous security remediation agent. Given an error log and buggy code, output ONLY the unified Git patch. Do not output conversational text.<|im_end|>
<|im_start|>user
Error: {item['error_log']}
Code:
{item['buggy_code']}<|im_end|>
<|im_start|>assistant
````diff
{item['patch']}
```<|im_end|>"""
        formatted.append({"text": text})

    return Dataset.from_list(formatted)
# ```

# **Delete `formatting_prompts_func` entirely** — you don't need it anymore.

# **Update `SFTConfig`** — set `dataset_text_field="text"` so the trainer knows which column already holds ready-to-train text:

# ```python
# sft_config = SFTConfig(
#     output_dir=OUTPUT_DIR,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     logging_steps=1,
#     max_steps=20,
#     optim="paged_adamw_8bit",
#     fp16=True,
#     bf16=False,
#     save_strategy="no",
#     max_length=1024,
#     dataset_text_field="text",   # changed from None
#     packing=False,
# )
# ```

# **Update `SFTTrainer`** — remove `formatting_func` since the dataset is already formatted:

# ```python
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=dataset,
#     peft_config=peft_config,
#     args=sft_config,
# )
# ```

## Why this works better
# This removes TRL's internal formatting/batching logic from the picture entirely — the trainer just reads a plain `"text"` string column, which is the single most stable, version-independent interface across TRL releases. No more chasing API renames on `formatting_func`'s calling convention.

# Since `model` in memory is unaffected by this change (only `dataset`, `sft_config`, and `trainer` are being redefined), you can update Cell B only and **do not need to re-run Cell A** this time — no reload wait. Run it and share the next output.

# def formatting_prompts_func(example):
#     output_texts = []
#     for i in range(len(example['error_log'])):
#         prompt = f"""<|im_start|>system
# You are PatchPilot, an autonomous security remediation agent. Given an error log and buggy code, output ONLY the unified Git patch. Do not output conversational text.<|im_end|>
# <|im_start|>user
# Error: {example['error_log'][i]}
# Code:
# {example['buggy_code'][i]}<|im_end|>
# <|im_start|>assistant
# ```diff
# {example['patch'][i]}
# ```<|im_end|>"""
#         output_texts.append(prompt)
#     return output_texts

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

print("Attaching LoRA adapters...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# for name, param in model.named_parameters():
#     if param.requires_grad:
#         param.data = param.data.to(torch.float16)

dataset = generate_synthetic_training_data()

print("Initializing SFT Trainer...")
# training_args = TrainingArguments(
#     output_dir=OUTPUT_DIR,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     logging_steps=1,
#     max_steps=20,
#     optim="paged_adamw_8bit",
#     fp16=True,  # Fixed for Colab T4 GPU
#     bf16=False, # Fixed for Colab T4 GPU
#     save_strategy="no",
# )


sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    max_steps=20,
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=False,
    save_strategy="no",
    max_length=1024,              # moved here from TrainingArguments
    dataset_text_field="text",      # not needed since we use formatting_func
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    # peft_config=peft_config,
    args=sft_config
)

print("Beginning LoRA fine-tuning...")
trainer.train()

print("Saving adapters...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training Complete!")

Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Beginning LoRA fine-tuning...


Step,Training Loss
1,2.702847
2,2.457520
3,2.284711
4,2.149658
5,2.026206
6,1.919794
7,1.823501
8,1.723044
9,1.619645
10,1.522477


Saving adapters...
Training Complete!


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))

CUDA available: False
Device count: 0


In [ ]:
import shutil
from google.colab import files

# Compress the adapter directory
shutil.make_archive("patchpilot-lora-v1", 'zip', "patchpilot-lora-v1")

# Download to your computer
files.download("patchpilot-lora-v1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import trl
print(trl.__version__)

from trl import SFTConfig
import inspect
print([p for p in inspect.signature(SFTConfig.__init__).parameters if 'len' in p.lower() or 'seq' in p.lower()])

1.9.0
['length_column_name', 'max_length']


In [ ]:
import trl
print(trl.__version__)

1.9.0


## Testing the Fine-Tuned Model

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

# Load the base model in 4-bit quantization (same as training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load the LoRA adapters
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()

print("Model and tokenizer loaded for inference.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model and tokenizer loaded for inference.


In [8]:
# Example input for testing
test_error_log = "NullPointerException: Attempt to invoke virtual method on a null object reference"
test_buggy_code = "public void displayUser(User user) {\n    System.out.println(user.getName());\n}"

# Format the prompt exactly as in training
test_prompt = f"""<|im_start|>system\nYou are PatchPilot, an autonomous security remediation agent. Given an error log and buggy code, output ONLY the unified Git patch. Do not output conversational text.<|im_end|>\n<|im_start|>user\nError: {test_error_log}\nCode:\n{test_buggy_code}<|im_end|>\n<|im_start|>assistant\n```diff\n"""

# Encode the input
input_ids = tokenizer(test_prompt, return_tensors="pt").input_ids.cuda()

# Generate the patch
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=200, # Adjust as needed for patch length
        do_sample=False, # For deterministic output
        pad_token_id=tokenizer.eos_token_id # Ensure generation stops at EOS
    )

# Decode and print the output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

# Extract only the patch part, as the prompt is included in generated_text
# We're looking for the part after the assistant prompt starter
assistant_prefix = "<|im_start|>assistant\n```diff\n"
if assistant_prefix in generated_text:
    patch_start_index = generated_text.find(assistant_prefix) + len(assistant_prefix)
    patch_text = generated_text[patch_start_index:].strip()
    # Further strip potential trailing assistant markers
    if "<|im_end|>" in patch_text:
        patch_text = patch_text[:patch_text.find("<|im_end|>")].strip()
    if "```" in patch_text:
        patch_text = patch_text[:patch_text.find("```")].strip()
else:
    patch_text = "Could not extract patch. Full generated text:\n" + generated_text

print("Generated Patch:")
print(patch_text)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Generated Patch:
diff --git a/src/User.java b/src/User.java
--- a/src/User.java
+++ b/src/User.java
@@ -1,3 +1,3 @@
 public class User {
-    private String name;
+    private String name;
 
     public User(String name) {
         this.name = name;
